# Seed

In [1]:
random_s = 47

# Importations

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_predict
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier, VotingRegressor, StackingRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV, RFE
from sklearn.model_selection import StratifiedKFold

import xgboost as xgb

# Preprocessing the data

In [3]:
#Ouverture du fichier train.csv
data = pd.read_csv("train.csv", sep=",")

print(data.shape)

data.head()

(10906, 37)


,Id,Assvie,Sexe,Age,Couple,Statmat,Nat,Dip,Occ,Work,...,Capi,Pea,Urbani,Zres,Nbenf,Logt,Terre,Dette,Detlog,Detvo
0,0,N,H,39,N,cel,fra,tec,ina,catC,...,N,N,U6,idf,0,loc,N,N,N,N
1,1,O,H,60,O,mar,fra,bac+5,ret,catA,...,N,O,U6,idf,3,prop,N,N,N,N
2,2,O,F,60,O,mar,fra,bac+5,ret,catA,...,N,N,U6,idf,3,prop,N,N,N,N
3,3,N,H,37,N,cel,fra,bac+5,act,catA,...,N,N,U6,idf,0,prop,N,O,O,N
4,4,N,H,27,N,cel,fra,bac+2,act,NaN,...,N,N,U6,idf,0,loc,N,N,N,N


In [4]:
#print missing values

# missing values
total = data.isnull().sum().sort_values(ascending=False)

# show
pd.DataFrame(data={'Missing': total}).head(10)

,Missing
Work,2220
Gpm,1107
Gpp,1107
Mere,981
Dip,981
Pere,981
Evtgrav,909
Statpro,695
Id,0
Assdec,0


In [5]:
print(pd.unique(data['Work']))

['catC' 'catA' nan 'emp' 'Waut' 'catB']


In [6]:
#drop Id columns cause it is irrelevant 
data = data.drop(['Id'], axis = 1)

In [7]:
nominal = [] 
           
ordinal = ['Zres', 'Urbani', 'Occ', 'Nat', 'Statmat','Assvie', 'Sexe', 'Couple', 'Herit', 'Livep', 'Eplog', 'Epsal', 'Vmob', 'Livdf', 'Pel', 
           'Cel', 'Pep', 'Assdec', 'Retraite', 'Capi', 'Pea', 'Logt', 'Terre', 'Dette', 'Detlog', 
           'Detvo'] 
           
discrete = []

In [8]:
#Ordinal variables are converted into discrete features
for col in ordinal:
    v = list(data[col].values)
    data[col] = LabelEncoder().fit_transform(v)

In [9]:
#The missing columns have to be delt separatly
missing_col = ['Work', 'Gpm','Gpp', 'Pere', 'Dip', 'Mere', 'Evtgrav', 'Statpro']

In [10]:
best_mod = {}
best_mod['Work'] = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate = 0.1, min_samples_split = 8, 
                                                       subsample = 0.8,  max_features = 'sqrt', random_state = 42)
best_mod['Gpm'] = GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate = 0.2, min_samples_split = 5, 
                                                       subsample = 0.5,  max_features = 'sqrt', random_state = 42)
best_mod['Gpp'] = GradientBoostingClassifier(n_estimators=50, max_depth=4, learning_rate = 0.2, min_samples_split = 8, 
                                                       subsample = 0.8,  max_features = 'sqrt', random_state = 42)
best_mod['Pere'] = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate = 0.09, min_samples_split = 2, 
                                                       subsample = 1.0,  max_features = 'sqrt', random_state = 42)
best_mod['Dip'] = GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate = 0.2, min_samples_split = 5, 
                                                       subsample = 0.5,  max_features = 'sqrt', random_state = 42)
best_mod['Mere'] = GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate = 0.09, min_samples_split = 5, 
                                                       subsample = 0.5,  max_features = 'sqrt', random_state = 42)
best_mod['Evtgrav'] = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate = 0.09, min_samples_split = 2, 
                                                       subsample = 1.0,  max_features = 'sqrt', random_state = 42)
best_mod['Statpro'] = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate = 0.1, min_samples_split = 5, 
                                                       subsample = 1.0,  max_features = 'sqrt', random_state = 42)

In [11]:
#GradientBoostingClassifier for all missing values
for temp in missing_col:
    # Split data into train and test sets
    train_data = data.dropna(subset=[temp])
    test_data = data[data[temp].isna()]

    # Split features and target variable
    X_train = train_data.drop(columns=['Work', 'Gpm','Gpp', 'Pere', 'Dip', 'Mere', 'Evtgrav', 'Statpro'])
    v = list(train_data[temp].values)
    y_train = LabelEncoder().fit_transform(v)

    # Train random forest classifier
    clf = best_mod[temp]
    clf.fit(X_train, y_train)

    # Predict missing values
    X_test = test_data.drop(columns=['Work', 'Gpm','Gpp', 'Pere', 'Dip', 'Mere', 'Evtgrav', 'Statpro'])
    missing_values = clf.predict(X_test)

    # Replace missing values in original dataframe
    data.loc[data[temp].notna(), temp] = y_train
    data.loc[data[temp].isna(), temp] = missing_values

In [12]:
data.columns

Index(['Assvie', 'Sexe', 'Age', 'Couple', 'Statmat', 'Nat', 'Dip', 'Occ',
       'Work', 'Statpro', 'Herit', 'Pere', 'Mere', 'Gpp', 'Gpm', 'Evtgrav',
       'Livep', 'Eplog', 'Epsal', 'Vmob', 'Livdf', 'Pel', 'Cel', 'Pep',
       'Assdec', 'Retraite', 'Capi', 'Pea', 'Urbani', 'Zres', 'Nbenf', 'Logt',
       'Terre', 'Dette', 'Detlog', 'Detvo'],
      dtype='object')

# XGB

## Features Selection

### Assvie

In [13]:
# Load your data here
y = np.array(data['Assvie'])
X = np.array(data.drop(['Retraite', 'Assvie'], axis=1))

# Create an XGBoost model
xgb_model = xgb.XGBClassifier()

# Initialize RFE with XGBoost model as the estimator
xgb_rfe_model = RFE(estimator=xgb_model)

# Fit RFE model to the data
xgb_rfe_model_fit = xgb_rfe_model.fit(X, y)

# Extract the ranking of features from RFE
feat_index = pd.Series(data=xgb_rfe_model_fit.ranking_, index=data.drop(['Retraite', 'Assvie'], axis=1).columns)

# Get the significant features from RFE (features with ranking 1)
signi_feat_rfe = feat_index[feat_index == 1].index

# Print the significant features
print('Significant features from RFE:', signi_feat_rfe)

Significant features from RFE: Index(['Age', 'Statmat', 'Occ', 'Work', 'Statpro', 'Herit', 'Pere', 'Mere',
       'Livep', 'Eplog', 'Epsal', 'Vmob', 'Livdf', 'Pel', 'Pep', 'Assdec',
       'Pea'],
      dtype='object')


In [14]:
print('Original number of features present in the dataset : {}'.format(X.shape[1]))
print()
print('Number of features selected by the Recursive feature selection technique is : {}'.format(len(signi_feat_rfe)))

Original number of features present in the dataset : 34

Number of features selected by the Recursive feature selection technique is : 17


In [15]:
features_A = ['Age', 'Statmat', 'Occ', 'Work', 'Statpro', 'Herit', 'Pere', 'Mere',
       'Livep', 'Eplog', 'Epsal', 'Vmob', 'Livdf', 'Pel', 'Pep', 'Assdec',
       'Pea']

### Retraite

In [16]:
# Load your data here
y = np.array(data['Retraite'])
X = np.array(data.drop(['Retraite', 'Assvie'], axis=1))

# Create an XGBoost model
xgb_model = xgb.XGBClassifier()

# Initialize RFE with XGBoost model as the estimator
xgb_rfe_model = RFE(estimator=xgb_model)

# Fit RFE model to the data
xgb_rfe_model_fit = xgb_rfe_model.fit(X, y)

# Extract the ranking of features from RFE
feat_index = pd.Series(data=xgb_rfe_model_fit.ranking_, index=data.drop(['Retraite', 'Assvie'], axis=1).columns)

# Get the significant features from RFE (features with ranking 1)
signi_feat_rfe = feat_index[feat_index == 1].index

# Print the significant features
print('Significant features from RFE:', signi_feat_rfe)

Significant features from RFE: Index(['Age', 'Statmat', 'Nat', 'Dip', 'Occ', 'Work', 'Statpro', 'Mere',
       'Livep', 'Epsal', 'Vmob', 'Pep', 'Assdec', 'Pea', 'Nbenf', 'Terre',
       'Detlog'],
      dtype='object')


In [17]:
print('Original number of features present in the dataset : {}'.format(X.shape[1]))
print()
print('Number of features selected by the Recursive feature selection technique is : {}'.format(len(signi_feat_rfe)))

Original number of features present in the dataset : 34

Number of features selected by the Recursive feature selection technique is : 17


In [18]:
features_R = ['Age', 'Statmat', 'Occ', 'Work', 'Statpro', 'Herit', 'Pere', 'Mere',
       'Livep', 'Eplog', 'Epsal', 'Vmob', 'Livdf', 'Pel', 'Pep', 'Assdec',
       'Pea']

## Find the best hyperparmeters

### Assvie

In [19]:
#data_features = data[features_R]
data_features = np.array(data.drop(['Retraite', 'Assvie'], axis=1))

X = np.array(data_features)
y = np.array(data['Assvie'])

In [20]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the hyperparameter grid to search
param_grid = {
    'booster': ['gbtree', 'gblinear'],
    'learning_rate': [0.01],
    'n_estimators': [500],
    'max_depth': [4],
    'gamma': [0],
    'subsample': [0.8],
    'colsample_bytree': [1.0],
    'min_child_weight': [1],
    'scale_pos_weight': [2],
    'random_state' : [47]
    # Add more hyperparameters to tune as needed
}

# Create an XGBoost model
xgb_model = xgb.XGBClassifier()

# Perform grid search using GridSearchCV with roc_auc scoring
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='roc_auc', cv=5,n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best hyperparameters and the corresponding model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Train the best model on the training data
best_model.fit(X_train, y_train)

# Make probability predictions on the testing data
y_pred_proba = best_model.predict_proba(X_test)[:, 1]  # Use the predicted class probabilities for the positive class

# Calculate the ROC AUC score of the model on the testing data
roc_auc = roc_auc_score(y_test, y_pred_proba)
print("Best Hyperparameters:", best_params)
print("ROC AUC Score:", roc_auc)

Best Hyperparameters: {'booster': 'gbtree', 'colsample_bytree': 1.0, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 500, 'random_state': 47, 'scale_pos_weight': 2, 'subsample': 0.8}
ROC AUC Score: 0.769737545878751


In [21]:
from sklearn.calibration import CalibratedClassifierCV

# Train the best model on the training data
best_model.fit(X_train, y_train)

# Create a calibrated classifier with Platt Scaling
calibrated_clf = CalibratedClassifierCV(best_model, method='sigmoid', cv='prefit')
calibrated_clf.fit(X_train, y_train)

# Make probability predictions on the testing data
y_pred_proba_calibrated = calibrated_clf.predict_proba(X_test)[:, 1]

# Calculate the ROC AUC score of the calibrated model on the testing data
roc_auc_calibrated = roc_auc_score(y_test, y_pred_proba_calibrated)
print("ROC AUC Score (Calibrated):", roc_auc_calibrated)

ROC AUC Score (Calibrated): 0.769737545878751


In [22]:
model_Assvie = calibrated_clf

### Retraite

In [23]:
#data_features = data[features_R]
data_features = np.array(data.drop(['Retraite', 'Assvie'], axis=1))

X = np.array(data_features)
y = np.array(data['Retraite'])

In [24]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the hyperparameter grid to search
param_grid = {
    'booster': ['gbtree', 'gblinear'],
    'learning_rate': [ 0.01],
    'n_estimators': [1000, 2000],
    'max_depth': [3],
    'min_child_weight': [2],
    'gamma': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'scale_pos_weight': [1],
    'random_state' : [88]
    # Add more hyperparameters to tune as needed
}

# Create an XGBoost model
xgb_model = xgb.XGBClassifier()

# Perform grid search using GridSearchCV with roc_auc scoring
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='roc_auc', cv=5,n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best hyperparameters and the corresponding model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Train the best model on the training data
best_model.fit(X_train, y_train)

# Make probability predictions on the testing data
y_pred_proba = best_model.predict_proba(X_test)[:, 1]  # Use the predicted class probabilities for the positive class

# Calculate the ROC AUC score of the model on the testing data
roc_auc = roc_auc_score(y_test, y_pred_proba)
print("Best Hyperparameters:", best_params)
print("ROC AUC Score:", roc_auc)

Best Hyperparameters: {'booster': 'gbtree', 'colsample_bytree': 0.8, 'gamma': 0.1, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 2, 'n_estimators': 1000, 'random_state': 88, 'scale_pos_weight': 1, 'subsample': 0.8}
ROC AUC Score: 0.8470969932470668


In [25]:
from sklearn.calibration import CalibratedClassifierCV

# Train the best model on the training data
best_model.fit(X_train, y_train)

# Create a calibrated classifier with Platt Scaling
calibrated_clf = CalibratedClassifierCV(best_model, method='sigmoid', cv='prefit')
calibrated_clf.fit(X_train, y_train)

# Make probability predictions on the testing data
y_pred_proba_calibrated = calibrated_clf.predict_proba(X_test)[:, 1]

# Calculate the ROC AUC score of the calibrated model on the testing data
roc_auc_calibrated = roc_auc_score(y_test, y_pred_proba_calibrated)
print("ROC AUC Score (Calibrated):", roc_auc_calibrated)

ROC AUC Score (Calibrated): 0.8470969932470668


In [26]:
model_Retraite = calibrated_clf

# Predictions

In [27]:
df = pd.read_csv("test.csv")

df = df.drop(['Id'], axis = 1)

nominal = [] 
           
ordinal = ['Sexe', 'Couple', 'Herit', 'Livep', 'Eplog', 'Epsal', 'Vmob', 'Livdf', 'Pel', 
           'Cel', 'Pep', 'Assdec', 'Capi', 'Pea', 'Logt', 'Terre', 'Dette', 'Detlog', 
           'Detvo', 'Zres', 'Urbani', 'Occ', 'Nat', 'Statmat']
           
discrete = []

#Ordinal variables are converted into discrete features
for col in ordinal:
    v = list(df[col].values)
    df[col] = LabelEncoder().fit_transform(v)
    
missing_col = ['Work', 'Gpm','Gpp', 'Pere', 'Dip', 'Mere', 'Evtgrav', 'Statpro']

In [28]:
for temp in missing_col:
    # Split data into train and test sets
    train_df = df.dropna(subset=[temp])
    test_df = df[df[temp].isna()]

    # Split features and target variable
    X_train = train_df.drop(columns=['Work', 'Gpm','Gpp', 'Pere', 'Dip', 'Mere', 'Evtgrav', 'Statpro'])
    v = list(train_df[temp].values)
    y_train = LabelEncoder().fit_transform(v)

    # Train random forest classifier
    clf = best_mod[temp]
    clf.fit(X_train, y_train)

    # Predict missing values
    X_test = test_df.drop(columns=['Work', 'Gpm','Gpp', 'Pere', 'Dip', 'Mere', 'Evtgrav', 'Statpro'])
    missing_values = clf.predict(X_test)

    # Replace missing values in original dataframe
    df.loc[df[temp].notna(), temp] = y_train
    df.loc[df[temp].isna(), temp] = missing_values

In [29]:
df_A = df[features_A]
df_R = df[features_R]

#X_A = np.array(df_A)
#X_R = np.array(df_R)

X_A = np.array(df)
X_R = np.array(df)

In [30]:
sub = pd.read_csv('sample_submission.csv')

predictions=model_Assvie.predict_proba(X_A)[:, 1]
#predictions=model.predict(X_A)
sub['Assvie'] = predictions

predictions=model_Retraite.predict_proba(X_R)[:, 1]
sub['Retraite'] = predictions

sub.to_csv('CSV/XGB.csv', index=False)